# 🤟 Step 1: ISL Video-to-Landmark Feature Extractor
**Project:** Real-Time Sign Language AI System (BiLSTM + Attention)

This notebook converts your downloaded Indian Sign Language `.mp4` videos into uniform `(60, 138)` MediaPipe Holistic `.npy` sequence files ready for model training.

- **Feature Vector:** 138 features per frame (Left Hand: 63 + Right Hand: 63 + Upper Pose: 12)
- **Temporal Window:** Exactly 60 frames (~2.0 seconds at 30 FPS)
- **Augmentation:** Generates spatial & temporal variations so single-sample dictionary signs produce 10–15 training sequences per class.

### 📦 Cell 1: Install Dependencies

In [ ]:
!pip install -q mediapipe opencv-python-headless numpy tqdm matplotlib

### ☁️ Cell 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### ⚙️ Cell 3: Configuration & Target Classes

In [ ]:
import os
import sys
import glob
import re
import cv2
import numpy as np
import mediapipe as mp
from tqdm.notebook import tqdm

# Path where your downloaded ISL .mp4 videos are stored in Google Drive
INPUT_VIDEOS_DIR = '/content/drive/MyDrive/ISL_Videos'

# Output directory for extracted .npy sequences
OUTPUT_DATASET_DIR = '/content/drive/MyDrive/SignLanguageAI/dataset'

TARGET_FRAMES = 60
TARGET_FEATURES = 138  # Left Hand (63) + Right Hand (63) + Upper Pose (12)
AUGMENTATIONS_PER_VIDEO = 12  # Generates 12 augmented variations per video

# 'TOP_35' for fast high-accuracy demo, or 'ALL' for entire dictionary
EXTRACTION_MODE = 'TOP_35'

TOP_35_CLASSES = [
    'Thank You', 'Help', 'Sorry', 'Welcome', 'Good-1',
    'Water', 'Food', 'Eat', 'Drink', 'Father',
    'Mother-1', 'Brother', 'Sister', 'Family', 'Friend',
    'School', 'Doctor', 'Hospital', 'Teacher', 'Book',
    'Money', 'House', 'Home', 'Work', 'Time',
    'Day', 'Sun', 'Moon', 'Car', 'Bus',
    'Baby', 'Boy', 'Girl', 'Dog', 'Cat-1'
]

print(f'[*] Extraction Mode: {EXTRACTION_MODE}')
print(f'[*] Input Directory: {INPUT_VIDEOS_DIR}')
print(f'[*] Output Directory: {OUTPUT_DATASET_DIR}')
if EXTRACTION_MODE == 'TOP_35':
    print(f'[*] Target Classes ({len(TOP_35_CLASSES)}): {TOP_35_CLASSES}')


### 🧠 Cell 4: Landmark Extraction Functions

In [ ]:
mp_holistic = mp.solutions.holistic

def extract_holistic_landmarks(results) -> np.ndarray:
    # Left Hand (21 * 3 = 63)
    if results.left_hand_landmarks:
        lh = np.array([[lm.x, lm.y, lm.z] for lm in results.left_hand_landmarks.landmark]).flatten()
    else:
        lh = np.zeros(21 * 3, dtype=np.float32)

    # Right Hand (21 * 3 = 63)
    if results.right_hand_landmarks:
        rh = np.array([[lm.x, lm.y, lm.z] for lm in results.right_hand_landmarks.landmark]).flatten()
    else:
        rh = np.zeros(21 * 3, dtype=np.float32)

    # Upper Body Pose (11, 12, 13, 14: shoulders and elbows -> 4 * 3 = 12)
    if results.pose_landmarks:
        pose_lms = results.pose_landmarks.landmark
        pose_coords = []
        for idx in [11, 12, 13, 14]:
            if idx < len(pose_lms):
                pose_coords.extend([pose_lms[idx].x, pose_lms[idx].y, pose_lms[idx].z])
            else:
                pose_coords.extend([0.0, 0.0, 0.0])
        pose = np.array(pose_coords, dtype=np.float32)
    else:
        pose = np.zeros(4 * 3, dtype=np.float32)

    return np.concatenate([lh, rh, pose])  # Shape: (138,)

def sample_frame_indices(total_frames: int, target_frames: int = 60) -> np.ndarray:
    if total_frames <= 0:
        return np.zeros(target_frames, dtype=int)
    if total_frames >= target_frames:
        return np.linspace(0, total_frames - 1, target_frames, dtype=int)
    else:
        return np.round(np.linspace(0, total_frames - 1, target_frames)).astype(int)

def process_video_to_raw_sequence(video_path: str, holistic) -> np.ndarray:
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None
    raw_frames = []
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image_rgb.flags.writeable = False
        results = holistic.process(image_rgb)
        raw_frames.append(extract_holistic_landmarks(results))
    cap.release()
    if not raw_frames:
        return None
    raw_arr = np.array(raw_frames, dtype=np.float32)
    indices = sample_frame_indices(len(raw_arr), TARGET_FRAMES)
    return raw_arr[indices]  # Shape: (60, 138)


### 🔄 Cell 5: Data Augmentation Generator

In [ ]:
def augment_sequence(seq: np.ndarray, aug_idx: int) -> np.ndarray:
    augmented = seq.copy()
    nonzero_mask = (augmented != 0.0)
    # Gaussian coordinate jitter
    noise_scale = 0.008 * (1.0 + (aug_idx % 3) * 0.5)
    noise = np.random.normal(0.0, noise_scale, size=augmented.shape).astype(np.float32)
    augmented[nonzero_mask] += noise[nonzero_mask]
    # Spatial scale factor
    scale = np.random.uniform(0.92, 1.08)
    augmented[nonzero_mask] *= scale
    # Temporal speed warp
    if np.random.rand() > 0.3:
        curve = np.linspace(0, TARGET_FRAMES - 1, TARGET_FRAMES)
        warp = np.sin(np.linspace(0, np.pi, TARGET_FRAMES)) * np.random.uniform(-3, 3)
        warped_idx = np.clip(np.round(curve + warp), 0, TARGET_FRAMES - 1).astype(int)
        augmented = augmented[warped_idx]
    return augmented


### 🚀 Cell 6: Run Feature Extraction Pipeline

In [ ]:
os.makedirs(OUTPUT_DATASET_DIR, exist_ok=True)
all_videos = glob.glob(os.path.join(INPUT_VIDEOS_DIR, '*.mp4'))
print(f'[*] Total videos found: {len(all_videos)}')

target_video_map = {}
for vpath in all_videos:
    fsize = os.path.getsize(vpath)
    if fsize > 3.0 * 1024 * 1024 or fsize < 50 * 1024:
        continue  # Skip story/corrupted videos
    raw_name = os.path.splitext(os.path.basename(vpath))[0]
    base_clean = re.sub(r'_\d+$', '', raw_name).strip()
    if EXTRACTION_MODE == 'TOP_35':
        for tc in TOP_35_CLASSES:
            if tc.lower() in [base_clean.lower(), raw_name.lower()]:
                target_video_map.setdefault(tc, []).append(vpath)
                break
    else:
        target_video_map.setdefault(base_clean, []).append(vpath)

print(f'[*] Processing {len(target_video_map)} classes...')
holistic = mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5, model_complexity=1)
total_saved = 0
class_counts = {}

for cls_name, vpaths in tqdm(target_video_map.items(), desc='Extracting Signs'):
    safe_cls = re.sub(r'[\\/*?:"<>|]', '_', cls_name)
    cls_dir = os.path.join(OUTPUT_DATASET_DIR, safe_cls)
    os.makedirs(cls_dir, exist_ok=True)
    class_counts[cls_name] = 0
    for v_idx, vpath in enumerate(vpaths):
        seq = process_video_to_raw_sequence(vpath, holistic)
        if seq is None or seq.shape != (TARGET_FRAMES, TARGET_FEATURES):
            continue
        np.save(os.path.join(cls_dir, f'{safe_cls}_v{v_idx}_orig.npy'), seq)
        class_counts[cls_name] += 1
        total_saved += 1
        for aug_i in range(AUGMENTATIONS_PER_VIDEO):
            aug_seq = augment_sequence(seq, aug_i)
            np.save(os.path.join(cls_dir, f'{safe_cls}_v{v_idx}_aug{aug_i}.npy'), aug_seq)
            class_counts[cls_name] += 1
            total_saved += 1

holistic.close()
print(f'\n[SUCCESS] Extracted {total_saved} total sequence files across {len(class_counts)} classes!')
for c, cnt in sorted(class_counts.items()):
    print(f'  - {c:<20s} : {cnt:3d} .npy files')


### 💤 Cell 7: Generate Idle (Resting Pose) Class
This adds synthetic resting posture samples so your real-time detector will not trigger false positives when the user rests their hands.

In [ ]:
idle_dir = os.path.join(OUTPUT_DATASET_DIR, 'idle')
os.makedirs(idle_dir, exist_ok=True)
# Create 50 idle sequences with hands at resting neutral position
for i in range(50):
    idle_seq = np.zeros((TARGET_FRAMES, TARGET_FEATURES), dtype=np.float32)
    # Add small resting shoulders/elbows pose landmarks (normalized coords around center)
    left_sh = [0.6 + np.random.normal(0, 0.01), 0.5 + np.random.normal(0, 0.01), 0.0]
    right_sh = [0.4 + np.random.normal(0, 0.01), 0.5 + np.random.normal(0, 0.01), 0.0]
    left_el = [0.65 + np.random.normal(0, 0.01), 0.75 + np.random.normal(0, 0.01), 0.0]
    right_el = [0.35 + np.random.normal(0, 0.01), 0.75 + np.random.normal(0, 0.01), 0.0]
    pose_vals = np.array(left_sh + right_sh + left_el + right_el, dtype=np.float32)
    idle_seq[:, 126:138] = pose_vals
    np.save(os.path.join(idle_dir, f'idle_{i:03d}.npy'), idle_seq)
print(f'[*] Created 50 synthetic baseline sequences in {idle_dir}')
